# 03 · Tools & Search: How Agents Reach the Outside World

An LLM only knows its training data (frozen at a cutoff date). A *research* agent must fetch **current, verifiable
information with sources**. In LangChain, external capabilities are exposed to the model as **tools**.

This notebook:
1. Explores each information source on its own: **Tavily** (search + page extraction), **DuckDuckGo**, **arXiv**, **Wikipedia**.
2. Normalises their very different outputs into **one `SearchResult` schema**, so agents don't care which provider answered.
3. Compares them on latency, result count and content richness.
4. Wraps them as **LangChain tools** with **fallbacks** and **error handling**, exactly as the researcher agent will use them.

Search costs (free tiers): Tavily ≈1 credit per basic search, 2 per advanced, 1 per 5 extracted URLs
(1,000 credits/month). arXiv, Wikipedia and DuckDuckGo are free and keyless.

## 0 · Setup

In [1]:
from nb_setup import setup, timer
ROOT = setup()

import time
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field

API keys:
  GROQ_API_KEY         MISSING
  OPENROUTER_API_KEY   set
  TAVILY_API_KEY       set
  LANGSMITH_API_KEY    set
  LangSmith tracing    true


LLM cache: data\llm_cache.sqlite


## 1 · One schema for every source

Each provider returns differently-shaped data (Tavily: `content`/`score`; DDG: `body`/`href`; arXiv: `summary`/`entry_id`...).
If agents consumed raw provider output, every prompt and every downstream step would need provider-specific logic.

Instead we apply the **Adapter pattern**: each provider gets a small function that converts its output into a common
`SearchResult`. Downstream code (researchers, fact-checker, citation formatter) only ever sees `SearchResult`.
This also gives every claim a `url` we can cite later.

In [2]:
class SearchResult(BaseModel):
    """Provider-independent search hit; the unit of evidence that flows through the system."""
    title: str
    url: str
    content: str = Field(description="Snippet or extracted text")
    source: Literal["tavily", "duckduckgo", "arxiv", "wikipedia"]
    score: float | None = Field(default=None, description="Provider relevance score, if any")
    published: str | None = Field(default=None, description="Publication date (ISO), if known")

## 2 · Tavily: search engine built for LLM agents

Unlike a normal search API (which returns 10 blue links + short snippets), Tavily returns **pre-cleaned, relevant text
chunks** per result plus a relevance `score`, which is ideal input for an LLM. Useful options:
- `max_results`: how many hits (more = more context tokens).
- `search_depth`: `"basic"` (fast, 1 credit) or `"advanced"` (better chunks, 2 credits).
- `topic`: `"general"` or `"news"`; `time_range`: `"day" | "week" | "month" | "year"` for recency.
- `include_answer`: Tavily's own short LLM-generated answer (we **don't** rely on it: we want primary sources).

In [3]:
from langchain_tavily import TavilySearch, TavilyExtract


def tavily_search(query: str, k: int = 5, depth: str = "basic", **kwargs) -> list[SearchResult]:
    raw = TavilySearch(max_results=k, search_depth=depth, **kwargs).invoke({"query": query})
    return [SearchResult(title=r["title"], url=r["url"], content=r["content"],
                         source="tavily", score=r.get("score")) for r in raw["results"]]


with timer("tavily basic"):
    hits = tavily_search("How do LangGraph supervisor multi-agent architectures work?", k=3)
for h in hits:
    print(f"{h.score:.2f}  {h.title[:70]}\n      {h.url}\n      {h.content[:150]}...\n")

[tavily basic] 2.53s
0.81  LangGraph Multi-Agent — Supervisor Pattern Guide (2026) | MyEngineerin
      https://myengineeringpath.dev/genai-engineer/langgraph-multi-agent
      Section titled “What This Code Does”

1. Three sub-agents are created with `create_react_agent`, each with its own tools and system prompt
2. Node wra...

0.79  Building Multi-Agent Systems with LangGraph and Ollama: Architectures,
      https://medium.com/@diwakarkumar_18755/building-multi-agent-systems-with-langgraph-and-ollama-architectures-concepts-and-code-383d4c01e00c
      Use case: Workflow orchestration, where logic is centrally controlled.

## 3. Supervisor (Tool-Calling) Architecture

Here, the supervisor uses a lang...

0.72  LangGraph Advanced – Build Hierarchical Multi-Level Supervisor Archite
      https://lilys.ai/en/notes/langgraph-swarm-20260202/langgraph-hierarchical-supervisor-swarm-ai-agents
      1. The top-level supervisor then moved the flow to the Portfolio Supervisor .
   2. The Portfol

### Recency: `topic="news"` + `time_range`
For trend questions ("latest", "in 2026"), restricting to recent news avoids outdated pages.

In [4]:
with timer("tavily news/month"):
    news = tavily_search("AI coding agents software engineering", k=3, topic="news", time_range="month")
for h in news:
    print(f"- {h.title[:90]}  ({h.url.split('/')[2]})")

[tavily news/month] 3.50s
- What Is Agentic Engineering? Build Software With AI Agents  (www.jamasoftware.com)
- Building a gated review pipeline for AI coding agents on AWS | AWS DevOps & Developer Prod  (aws.amazon.com)
- Codex & AI Coding Agents - Learn Agentic Coding  (www.youtube.com)


### Extraction: reading the full page
Search snippets are short. When a researcher finds a promising source, it can read the page. `TavilyExtract`
returns cleaned main text (no menus/ads). Pages can be **huge** (tens of thousands of characters, i.e. thousands
of tokens), so we always **truncate** before handing text to an LLM: this is our first context-budget control.

In [5]:
def tavily_extract(url: str, max_chars: int = 4000) -> str:
    raw = TavilyExtract().invoke({"urls": [url]})
    if not raw.get("results"):
        return f"Could not extract {url}: {raw.get('failed_results')}"
    text = raw["results"][0]["raw_content"]
    print(f"full page: {len(text):,} chars (~{len(text)//4:,} tokens) -> truncated to {max_chars:,}")
    return text[:max_chars]


with timer("tavily extract"):
    page = tavily_extract(hits[0].url)
print(page[:500])

full page: 32,548 chars (~8,137 tokens) -> truncated to 4,000
[tavily extract] 1.41s
[Skip to content](#_top)

[GitHub](https://github.com/mohitsaxenacs/)[LinkedIn](https://linkedin.com/in/mohitsaxena21)

# LangGraph Multi-Agent — Supervisor Pattern Guide (2026)

This **LangGraph multi-agent** guide walks through the supervisor pattern — where a central agent routes tasks to specialized sub-agents for research, code generation, and review. You will build a complete working system with state management, message routing, error handling, and production deployment patterns.

## 1. W


## 3 · DuckDuckGo: keyless web search (fallback candidate)

`ddgs` scrapes public search engines without an API key. Attractive as a free fallback, but scraping is fragile:
engines throttle or block automated traffic. We test it honestly and catch failures.

In [6]:
from ddgs import DDGS


def ddg_search(query: str, k: int = 5) -> list[SearchResult]:
    raw = DDGS(timeout=10).text(query, max_results=k)
    return [SearchResult(title=r["title"], url=r["href"], content=r["body"], source="duckduckgo") for r in raw]


try:
    with timer("duckduckgo"):
        ddg_hits = ddg_search("LangGraph supervisor multi-agent", k=3)
    for h in ddg_hits:
        print("-", h.title[:80], h.url)
except Exception as e:
    print(f"DuckDuckGo failed: {type(e).__name__}: {e}")

[duckduckgo] 5.68s
- langgraph_supervisor | LangChain Reference https://reference.langchain.com/python/langgraph-supervisor
- LangGraph Multi-Agent — Supervisor Pattern Guide (2026) https://myengineeringpath.dev/genai-engineer/langgraph-multi-agent/
- GitHub - langchain-ai/langgraph-supervisor-py https://github.com/langchain-ai/langgraph-supervisor-py


## 4 · arXiv: academic papers

arXiv's API is free and returns title, authors, abstract, date and a stable URL. Two lessons:
- arXiv search is **keyword-based**, not semantic: query phrasing matters a lot.
- Default sort is relevance; for "recent research" questions, sorting by submission date surfaces new work.

In [7]:
import arxiv


def arxiv_search(query: str, k: int = 3, recent: bool = False) -> list[SearchResult]:
    sort = arxiv.SortCriterion.SubmittedDate if recent else arxiv.SortCriterion.Relevance
    papers = arxiv.Client().results(arxiv.Search(query=query, max_results=k, sort_by=sort))
    return [SearchResult(title=p.title, url=p.entry_id, content=p.summary.replace("\n", " "),
                         source="arxiv", published=p.published.date().isoformat()) for p in papers]


for recent in (False, True):
    with timer(f"arxiv recent={recent}"):
        papers = arxiv_search("LLM multi-agent collaboration", k=3, recent=recent)
    for p in papers:
        print(f"  {p.published}  {p.title[:85]}")

[arxiv recent=False] 1.63s
  2024-10-16  MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collab
  2024-12-06  Towards Effective GenAI Multi-Agent Collaboration: Design and Evaluation for Enterpri
  2022-03-16  A Survey of Multi-Agent Deep Reinforcement Learning with Communication


[arxiv recent=True] 2.13s
  2026-11-17  Ambient LLMs: Data-Efficient Supervised Fine-tuning via Temperature Matching
  2026-11-17  Equilibrium Statistics and Stability in Random Two-Population and Three-Population Ev
  2026-11-17  Do LLM Agents Still Need to Be Told How to Self-Improve?


## 5 · Wikipedia: background and definitions

Ideal for the "encyclopedic" sub-questions the Planner produces ("what is X?"). Reliable, but not current and not a
primary source, so the fact-checker will weight it lower for time-sensitive claims.

In [8]:
from langchain_community.utilities import WikipediaAPIWrapper

_wiki = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1500)


def wikipedia_search(query: str) -> list[SearchResult]:
    return [SearchResult(title=d.metadata["title"], url=d.metadata["source"],
                         content=d.page_content, source="wikipedia") for d in _wiki.load(query)]


with timer("wikipedia"):
    wiki_hits = wikipedia_search("Retrieval-augmented generation")
for h in wiki_hits:
    print(f"- {h.title}: {h.content[:160]}...")

[wikipedia] 9.02s
- Retrieval-augmented generation: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sou...
- Vector database: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically i...


## 6 · Head-to-head comparison

Same two queries for every provider: one web-style and one academic-style. We record latency, number of results and
average content length (a proxy for how much useful text an agent receives). Failures are recorded, not raised.

In [9]:
PROVIDERS = {
    "tavily": lambda q: tavily_search(q, k=5),
    "duckduckgo": lambda q: ddg_search(q, k=5),
    "arxiv": lambda q: arxiv_search(q, k=5),
    "wikipedia": wikipedia_search,
}
QUERIES = ["small language models on-device deployment", "evaluation of retrieval-augmented generation"]

rows = []
for name, fn in PROVIDERS.items():
    for q in QUERIES:
        t0 = time.perf_counter()
        try:
            res = fn(q)
            rows.append({"provider": name, "query": q[:30], "seconds": round(time.perf_counter() - t0, 1),
                         "n_results": len(res),
                         "avg_chars": int(sum(len(r.content) for r in res) / max(len(res), 1)), "error": None})
        except Exception as e:
            rows.append({"provider": name, "query": q[:30], "seconds": round(time.perf_counter() - t0, 1),
                         "n_results": 0, "avg_chars": 0, "error": type(e).__name__})

comparison = pd.DataFrame(rows)
comparison.to_csv(ROOT / "data" / "03_search_comparison.csv", index=False)
comparison

,provider,query,seconds,n_results,avg_chars,error
0,tavily,small language models on-devic,5.0,5,1122,None
1,tavily,evaluation of retrieval-augmen,3.3,5,1217,None
2,duckduckgo,small language models on-devic,4.8,5,236,None
3,duckduckgo,evaluation of retrieval-augmen,7.3,5,678,None
4,arxiv,small language models on-devic,4.5,5,1111,None
5,arxiv,evaluation of retrieval-augmen,2.1,5,1281,None
6,wikipedia,small language models on-devic,8.9,2,1500,None
7,wikipedia,evaluation of retrieval-augmen,7.5,2,1500,None


## 7 · From functions to LangChain tools

A **tool** = a function + a **name** + a **description** + an **argument schema**. The model never sees our Python
code; it only sees the name/description/schema and decides *whether* and *how* to call it. So tool docstrings are,
again, prompt engineering: they tell the model *when* to use each tool.

Design rules for agent tools:
1. **Return text the LLM can read**, compact and numbered, with URLs so claims can be cited.
2. **Never raise.** A crashed tool crashes the agent; an error *message* lets the agent adapt (try another tool).
3. **Bound the output size** (context budget).
4. **Fallback inside the tool** where a substitute exists (Tavily → DuckDuckGo).

In [10]:
from langchain_core.tools import tool


def format_results(results: list[SearchResult], max_chars: int = 600) -> str:
    """Render results as a compact numbered list the LLM can cite from."""
    if not results:
        return "No results found."
    return "\n\n".join(
        f"[{i}] {r.title}\nURL: {r.url}" + (f"\nPublished: {r.published}" if r.published else "")
        + f"\n{r.content[:max_chars]}" for i, r in enumerate(results, 1))


def search_with_fallback(query: str, providers: list) -> tuple[str, list[SearchResult]]:
    """Try providers in order; return (provider_name, results) from the first that succeeds."""
    errors = []
    for name, fn in providers:
        try:
            results = fn(query)
            if results:
                return name, results
            errors.append(f"{name}: no results")
        except Exception as e:
            errors.append(f"{name}: {type(e).__name__}")
    raise RuntimeError("; ".join(errors))


WEB_PROVIDERS = [("tavily", lambda q: tavily_search(q, k=5)), ("duckduckgo", lambda q: ddg_search(q, k=5))]


@tool
def web_search(query: str) -> str:
    """Search the web for current information: news, documentation, blogs, company announcements.
    Use specific queries including entities and timeframes."""
    try:
        _, results = search_with_fallback(query, WEB_PROVIDERS)
        return format_results(results)
    except Exception as e:
        return f"ERROR: web search unavailable ({e}). Try search_papers or search_wikipedia instead."


@tool
def search_papers(query: str) -> str:
    """Search arXiv for academic research papers. Use for scientific methods, benchmarks and empirical results."""
    try:
        return format_results(arxiv_search(query, k=4), max_chars=800)
    except Exception as e:
        return f"ERROR: arXiv unavailable ({type(e).__name__}). Try web_search instead."


@tool
def search_wikipedia(query: str) -> str:
    """Look up background knowledge and definitions on Wikipedia. Not for recent events."""
    try:
        return format_results(wikipedia_search(query), max_chars=1000)
    except Exception as e:
        return f"ERROR: Wikipedia unavailable ({type(e).__name__})."


@tool
def fetch_page(url: str) -> str:
    """Read the main text of a web page found via search, when the snippet is not detailed enough."""
    try:
        return tavily_extract(url, max_chars=4000)
    except Exception as e:
        return f"ERROR: could not fetch page ({type(e).__name__})."


TOOLS = [web_search, search_papers, search_wikipedia, fetch_page]

# This is exactly what the LLM sees for each tool:
for t in TOOLS:
    print(f"{t.name}{list(t.args)}: {t.description.splitlines()[0]}")

web_search['query']: Search the web for current information: news, documentation, blogs, company announcements.
search_papers['query']: Search arXiv for academic research papers. Use for scientific methods, benchmarks and empirical results.
search_wikipedia['query']: Look up background knowledge and definitions on Wikipedia. Not for recent events.
fetch_page['url']: Read the main text of a web page found via search, when the snippet is not detailed enough.


In [11]:
# Tools are Runnables: invoke with a dict of arguments, exactly as the agent runtime will.
print(web_search.invoke({"query": "LangGraph Send API map-reduce"})[:900])

[1] Map-Reduce with the Send() API in LangGraph
URL: https://medium.com/ai-engineering-bootcamp/map-reduce-with-the-send-api-in-langgraph-29b92078b47d
Edges connect map nodes to the reducer node(s), forming the shuffle stage. LangGraph handles the routing automatically, ensuring all intermediate results reach the correct reducer.

5. Send() — Parallelizing the Workflow

The `Send()` API dispatches all map nodes simultaneously. Once all map nodes complete, the results flow to the reducer node(s) without blocking execution. This is how LangGraph achieves true parallel Map-Reduce.

Key Takeaways: [...] Key Takeaways:

   `Send()` enables parallel mapping across nodes.
   Node state holds intermediate outputs.
   Edges define how data moves to r

[2] Map-Reduce with the Send() API in LangGraph
URL: https://www.youtube.com/watch?v=5iYV0q6eKbM
What Map-Reduce means in the context of LLM applic


## 8 · Proving the fallback works (fault injection)

We can't wait for Tavily to go down to test the fallback, so we **inject a fault**: a provider that always raises,
placed first in the chain. The chain should skip it, try the next provider, and when *everything* fails, return a
helpful error string rather than raising. This style of testing will become unit tests in Phase 3.

In [12]:
def broken_provider(query):
    raise ConnectionError("simulated outage")

# Case 1: primary down -> next provider answers
name, res = search_with_fallback("vector databases comparison", [("broken", broken_provider),
                                                                  ("tavily", lambda q: tavily_search(q, k=2))])
print("case 1 answered by:", name, "|", len(res), "results")

# Case 2: everything down -> the tool returns an actionable message (no exception reaches the agent)
WEB_PROVIDERS_BACKUP = WEB_PROVIDERS
WEB_PROVIDERS = [("broken", broken_provider), ("also_broken", broken_provider)]
print("case 2 tool output:", web_search.invoke({"query": "anything"}))
WEB_PROVIDERS = WEB_PROVIDERS_BACKUP

case 1 answered by: tavily | 2 results
case 2 tool output: ERROR: web search unavailable (broken: ConnectionError; also_broken: ConnectionError). Try search_papers or search_wikipedia instead.


## 9 · Observations & Decisions

*Recorded from the run on 2026-11-18. Raw table: `data/03_search_comparison.csv`.*

**Observations**
- **Tavily** is fast (~2–5 s) and returns relevance-scored, LLM-ready chunks (~1,200 chars per result).
  `TavilyExtract` returned a 32k-character page (~8k tokens) that still starts with navigation links, so extracted
  text must be **truncated** (and ideally cleaned) before it reaches an LLM.
- **DuckDuckGo (`ddgs`) is intermittent**: in an earlier probe every backend failed (timeouts / "no results"); in this
  run it answered in 5–7 s. Its snippets are 3–5× shorter (236–678 chars). Good as a best-effort fallback, not more.
- **arXiv** relevance sort mixes in old papers (a 2022 survey), while **date sort destroys relevance**: it returned
  "Equilibrium Statistics in Random Two-Population..." for an LLM-agents query. Relevance sort + a year in the query
  is the better strategy.
- **Wikipedia** is reliable for background but the slowest (~7–9 s) and not current.
- The fault-injection test shows the fallback chain and the "never raise" rule work as intended.

**Decisions**
1. Unified **`SearchResult`** schema → `src/research_system/schemas/` ; provider adapters → `src/research_system/tools/`.
2. Researcher toolset: `web_search` (Tavily → DuckDuckGo best-effort), `search_papers` (arXiv), `search_wikipedia`,
   `fetch_page` (Tavily Extract, truncated to 4,000 chars).
3. Tools **never raise**: they return `ERROR: ... try X instead` so the agent can re-route.
4. Results are formatted as **numbered entries with URLs** to make citation tracking possible.
5. Defaults: `search_depth="basic"` (cost), `max_results=5`; use `topic="news"` + `time_range` for trend questions.
6. arXiv: **relevance sort only**; the researcher prompt tells the agent to include years in academic queries.
7. Optional future fallback with a key: Exa or Brave Search API (same adapter interface).